# Фінальний проєкт: енергоспоживання навчального корпусу

**Коротке формулювання:** на основі щомісячних даних про споживання електроенергії навчальним корпусом за 2022–2026 роки спрогнозувати споживання на наступні 6 місяців і вибрати набір енергоефективних заходів, що мінімізує сумарні витрати та викиди CO₂ за умови бюджету не більше 500 тис. грн.

## Постановка задачі (шаблон курсу)

| № | Блок | Зміст у цьому проєкті |
|---|------|------------------------|
| 1 | **Об’єкт** | Навчальний корпус університету (енергоспоживання) |
| 2 | **Проблема** | Який портфель енергозберігаючих заходів впровадити за обмеженого бюджету |
| 3 | **Дані** | Місячний ряд січ. 2022 – вер. 2026: температура, світловий день, навчальні дні, опалення, кВт·год (синтетичні, реалістична сезонність) |
| 4 | **Модель** | Фактори → споживання; baseline LR, багатофакторна LR, Random Forest |
| 5 | **Прогноз** | Горизонт 6 міс. (жовтень 2026 – березень 2027); сценарії warm / base / cold |
| 6 | **Рішення** | Керовані змінні: набір заходів M1–M6 (вкл./викл.) |
| 7 | **Критерії й обмеження** | Економіка (ефект грн/рік) і екологія (тонн CO₂); бюджет ≤ 500 тис.; економія ≥ 15% |
| 8 | **Очікуваний результат** | Рекомендований портфель + як він змінюється при зміні бюджету / ваги екології / погоди |

| Етап | Зміст |
|------|--------|
| КТ 1 | Дані, пропуски, аномалії, похідні показники |
| КТ 2 | EDA та порівняння регресійних моделей |
| КТ 3 | Часовий ряд, baseline і прогноз на 6 місяців |
| КТ 4 | Оптимізація портфеля заходів (бюджет / CO₂) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from itertools import combinations

plt.rcParams['figure.figsize'] = (10, 4)
np.random.seed(42)
print('Бібліотеки завантажено.')

## КТ 1. Дані, пропуски та аномалії

Створюємо синтетичний місячний ряд (січень 2022 — вересень 2026) для навчального корпусу: температура, тривалість світлового дня, навчальні дні, ознака опалювального сезону та споживання електроенергії. Далі штучно додаємо пропуски й аномалії, очищуємо їх і рахуємо CO₂ та вартість.

In [ ]:
dates = pd.date_range(start='2022-01-01', end='2026-09-01', freq='MS')
n = len(dates)
months = dates.month

temp_base = np.array([-3.5, -2.0, 2.5, 9.5, 15.0, 18.5, 20.5, 19.8, 14.5, 8.5, 3.0, -1.5])
temperature = np.array([temp_base[m - 1] + np.random.normal(0, 1.5) for m in months])

daylight_base = np.array([8.5, 10.0, 11.8, 13.8, 15.3, 16.2, 15.8, 14.3, 12.5, 10.6, 8.9, 8.1])
daylight = np.array([daylight_base[m - 1] for m in months])

study_days_base = np.array([15, 20, 22, 21, 21, 15, 2, 0, 22, 22, 21, 20])
study_days = np.array([max(0, study_days_base[m - 1] + np.random.randint(-1, 2)) for m in months])

heating_season = np.where((months <= 3) | (months >= 10) | (temperature < 8), 1, 0)

energy_kwh = (
    18.0
    - 0.65 * temperature
    + 0.45 * study_days
    + 9.5 * heating_season
    - 0.4 * daylight
    + np.random.normal(0, 1.2, n)
) * 1000

df_raw = pd.DataFrame({
    'date': dates,
    'temperature_c': np.round(temperature, 1),
    'daylight_hours': np.round(daylight, 1),
    'study_days': study_days,
    'heating_season': heating_season,
    'energy_kwh': np.round(energy_kwh, 0)
})

# Штучні пропуски та аномалії для КТ 1
df_dirty = df_raw.copy()
df_dirty.loc[5, 'temperature_c'] = np.nan
df_dirty.loc[18, 'energy_kwh'] = np.nan
df_dirty.loc[33, 'study_days'] = np.nan
df_dirty.loc[12, 'energy_kwh'] = 95000   # завищене значення
df_dirty.loc[27, 'energy_kwh'] = -500    # від'ємне значення

df_dirty.to_csv('data.csv', index=False)
print('1. Файл data.csv створено.')
print('   Пропусків до очищення:', df_dirty.isna().sum().sum())
print('   Опис «брудних» даних:')
print(df_dirty.isna().sum())
df_dirty.head()

In [ ]:
df = pd.read_csv('data.csv', parse_dates=['date'])

# Аномалії: від'ємні / фізично нереалістичні значення споживання
anomalies_mask = (df['energy_kwh'] < 5000) | (df['energy_kwh'] > 60000)
print('Знайдено аномалій energy_kwh:', anomalies_mask.sum())
df.loc[anomalies_mask, 'energy_kwh'] = np.nan

# Лінійна інтерполяція пропусків у часовому ряді
df['temperature_c'] = df['temperature_c'].interpolate(method='linear')
df['study_days'] = df['study_days'].interpolate(method='linear').round()
df['energy_kwh'] = df['energy_kwh'].interpolate(method='linear')

# Похідні показники
df['co2_tons'] = (df['energy_kwh'] * 0.42) / 1000   # ~0.42 кг CO2 / кВт·год
df['cost_uah'] = df['energy_kwh'] * 7.5             # тариф 7.5 грн / кВт·год

df.to_csv('data_cleaned.csv', index=False)
print('Пропусків після очищення:', df.isna().sum().sum())
print('Збережено data_cleaned.csv')
df.describe().round(2)

## КТ 2. EDA та порівняння моделей

Візуалізуємо динаміку споживання та залежність від температури. Порівнюємо три моделі на останніх 12 місяцях (hold-out):
1. базова лінійна (лише температура);
2. багатофакторна лінійна;
3. Random Forest.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(df['date'], df['energy_kwh'] / 1000, marker='o', markersize=3,
             label='Споживання (тис. кВт·год)')
axes[0].set_title('Динаміка енергоспоживання корпусу (2022–2026)')
axes[0].set_xlabel('Дата')
axes[0].set_ylabel('тис. кВт·год')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].scatter(df['temperature_c'], df['energy_kwh'] / 1000,
                c=df['heating_season'], cmap='coolwarm', s=50)
axes[1].set_title('Залежність споживання від температури повітря')
axes[1].set_xlabel('Середня температура (°C)')
axes[1].set_ylabel('тис. кВт·год')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Кореляції факторів зі споживанням
corr_cols = ['temperature_c', 'daylight_hours', 'study_days', 'heating_season', 'energy_kwh']
corr = df[corr_cols].corr()
print('Кореляційна матриця:')
print(corr.round(3))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha='right')
ax.set_yticklabels(corr_cols)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title('Кореляції ознак')
plt.tight_layout()
plt.show()

In [ ]:
train_df = df.iloc[:-12]
test_df = df.iloc[-12:]
y_train, y_test = train_df['energy_kwh'], test_df['energy_kwh']

model_base = LinearRegression().fit(train_df[['temperature_c']], y_train)
pred_base = model_base.predict(test_df[['temperature_c']])

features_imp = ['temperature_c', 'study_days', 'heating_season', 'daylight_hours']
model_imp = LinearRegression().fit(train_df[features_imp], y_train)
pred_imp = model_imp.predict(test_df[features_imp])

model_rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(train_df[features_imp], y_train)
pred_rf = model_rf.predict(test_df[features_imp])

results_kt2 = pd.DataFrame({
    'Модель': [
        '1. Базова (тільки температура)',
        '2. Покращена (багатофакторна лінійна)',
        '3. Random Forest (нелінійна)'
    ],
    'MAE (кВт·год)': [
        mean_absolute_error(y_test, pred_base),
        mean_absolute_error(y_test, pred_imp),
        mean_absolute_error(y_test, pred_rf)
    ],
    'RMSE (кВт·год)': [
        np.sqrt(mean_squared_error(y_test, pred_base)),
        np.sqrt(mean_squared_error(y_test, pred_imp)),
        np.sqrt(mean_squared_error(y_test, pred_rf))
    ],
    'R2 Score': [
        r2_score(y_test, pred_base),
        r2_score(y_test, pred_imp),
        r2_score(y_test, pred_rf)
    ]
}).round(2)

print('=== КТ 2: Порівняння моделей на тестовій вибірці (останні 12 міс.) ===')
print(results_kt2.to_string(index=False))

best_kt2 = results_kt2.loc[results_kt2['MAE (кВт·год)'].idxmin(), 'Модель']
print(f'\nНайкраща за MAE: {best_kt2}')

plt.figure(figsize=(11, 4))
plt.plot(test_df['date'], y_test / 1000, 'k-o', label='Факт', markersize=5)
plt.plot(test_df['date'], pred_base / 1000, '--', label='Базова LR')
plt.plot(test_df['date'], pred_imp / 1000, '--', label='Багатофакторна LR')
plt.plot(test_df['date'], pred_rf / 1000, '-', label='Random Forest')
plt.title('КТ 2: прогноз vs факт на тестовому періоді')
plt.xlabel('Дата')
plt.ylabel('тис. кВт·год')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## КТ 3. Часовий ряд і прогнозування

Будуємо ознаки тренду та сезонності (`t`, середнє за місяцем, sin/cos). Порівнюємо **наївний сезонний baseline** (значення того ж місяця рік тому) з лінійною моделлю часового ряду на горизонті 6 місяців. Далі робимо прогноз на жовтень 2026 — березень 2027 з трьома сценаріями погоди.

In [ ]:
df['t'] = np.arange(len(df))
df['month'] = df['date'].dt.month
df['sin_m'] = np.sin(2 * np.pi * df['month'] / 12)
df['cos_m'] = np.cos(2 * np.pi * df['month'] / 12)

horizon = 6
ts_train = df.iloc[:-horizon].copy()
ts_test = df.iloc[-horizon:].copy()

# Наївний сезонний прогноз: той самий місяць рік тому
naive_preds = df['energy_kwh'].iloc[-horizon - 12: -12].values

month_means = ts_train.groupby('month')['energy_kwh'].mean().to_dict()
ts_train['month_avg'] = ts_train['month'].map(month_means)
ts_test['month_avg'] = ts_test['month'].map(month_means)

ts_features = ['t', 'month_avg', 'sin_m', 'cos_m']
ts_model = LinearRegression().fit(ts_train[ts_features], ts_train['energy_kwh'])
main_ts_preds = ts_model.predict(ts_test[ts_features])

results_kt3 = pd.DataFrame({
    'Модель': ['Наївний сезонний baseline', 'Лінійна модель часового ряду'],
    'MAE (кВт·год)': [
        mean_absolute_error(ts_test['energy_kwh'], naive_preds),
        mean_absolute_error(ts_test['energy_kwh'], main_ts_preds)
    ],
    'RMSE (кВт·год)': [
        np.sqrt(mean_squared_error(ts_test['energy_kwh'], naive_preds)),
        np.sqrt(mean_squared_error(ts_test['energy_kwh'], main_ts_preds))
    ],
    'R2 Score': [
        r2_score(ts_test['energy_kwh'], naive_preds),
        r2_score(ts_test['energy_kwh'], main_ts_preds)
    ]
}).round(2)

print('=== КТ 3: Перевірка моделей часового ряду (горизонт 6 міс.) ===')
print(results_kt3.to_string(index=False))

In [ ]:
future_dates = pd.date_range(start='2026-10-01', periods=6, freq='MS')
future_df = pd.DataFrame({'date': future_dates})
future_df['t'] = np.arange(len(df), len(df) + 6)
future_df['month'] = future_df['date'].dt.month
future_df['sin_m'] = np.sin(2 * np.pi * future_df['month'] / 12)
future_df['cos_m'] = np.cos(2 * np.pi * future_df['month'] / 12)

# Для фінального прогнозу навчаємося на всіх доступних даних
full_month_means = df.groupby('month')['energy_kwh'].mean().to_dict()
df['month_avg'] = df['month'].map(full_month_means)
future_df['month_avg'] = future_df['month'].map(full_month_means)

ts_model_full = LinearRegression().fit(df[ts_features], df['energy_kwh'])
future_df['forecast_base_kwh'] = ts_model_full.predict(future_df[ts_features]).round(0)
future_df['forecast_cold_kwh'] = (future_df['forecast_base_kwh'] * 1.12).round(0)
future_df['forecast_warm_kwh'] = (future_df['forecast_base_kwh'] * 0.90).round(0)

plt.figure(figsize=(11, 4.5))
plt.plot(df['date'].iloc[-24:], df['energy_kwh'].iloc[-24:] / 1000,
         label='Фактичні дані', color='black', marker='o', markersize=4)
plt.plot(ts_test['date'], naive_preds / 1000,
         label='Перевірка: Наївний прогноз', linestyle='--', color='gray')
plt.plot(ts_test['date'], main_ts_preds / 1000,
         label='Перевірка: Основна модель', linestyle='--', color='blue')
plt.plot(future_df['date'], future_df['forecast_base_kwh'] / 1000,
         label='Прогноз на 6 міс. (Базовий)', color='red', marker='s')
plt.fill_between(
    future_df['date'],
    future_df['forecast_warm_kwh'] / 1000,
    future_df['forecast_cold_kwh'] / 1000,
    color='red', alpha=0.15, label='Інтервал (тепла/холодна зима)'
)
plt.title('Прогноз споживання енергії на опалювальний сезон (Жовтень 2026 – Березень 2027)')
plt.xlabel('Дата')
plt.ylabel('тис. кВт·год')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

total_forecast_kwh = future_df['forecast_base_kwh'].sum()
print('Сумарний базовий прогноз на 6 міс.:', f'{total_forecast_kwh:,.0f}', 'кВт·год')
print('Холодна зима (+12%):', f"{future_df['forecast_cold_kwh'].sum():,.0f}", 'кВт·год')
print('Тепла зима (−10%):', f"{future_df['forecast_warm_kwh'].sum():,.0f}", 'кВт·год')
future_df[['date', 'forecast_warm_kwh', 'forecast_base_kwh', 'forecast_cold_kwh']]

## КТ 4. Вибір оптимального рішення

Перебираємо всі комбінації з 6 заходів з обмеженнями:
- бюджет ≤ `LIVE_BUDGET`;
- сумарне зниження споживання ≥ 15%;
- інтегральна оцінка = (1 − w)·економіка + w·екологія.

На захисті змінюйте `LIVE_BUDGET`, `LIVE_ECO_WEIGHT` і `WEATHER_SCENARIO`.

In [ ]:
measures = pd.DataFrame({
    'id': ['M1', 'M2', 'M3', 'M4', 'M5', 'M6'],
    'Назва заходу': [
        'Модернізація освітлення (LED + датчики руху)',
        'Автоматичне балансування системи опалення (ІТП)',
        'Смарт-термостати для аудиторій за розкладом',
        'Заміна та герметизація вікон у старих крилах',
        'Встановлення рекуператорів повітря у великих залах',
        'Дахова сонячна електростанція (СЕС 15 кВт)'
    ],
    'Вартість_впровадження_грн': [95000, 185000, 110000, 240000, 160000, 310000],
    'Економія_енергії_відсоток': [5.5, 11.0, 7.5, 8.5, 6.0, 9.0],
    'Додатковий_еко_бонус_тонн_CO2': [0.2, 0.5, 0.3, 0.4, 0.6, 2.2]
})

TARIFF_UAH = 7.5
CO2_FACTOR = 0.00042


def optimize_portfolio(budget_limit, eco_weight, scenario_kwh, min_saving_pct=15.0):
    valid_solutions = []
    n_items = len(measures)
    for r in range(1, n_items + 1):
        for comb in combinations(range(n_items), r):
            sub = measures.iloc[list(comb)]
            cost = sub['Вартість_впровадження_грн'].sum()
            saving_pct = sub['Економія_енергії_відсоток'].sum()
            if cost <= budget_limit and saving_pct >= min_saving_pct:
                saved_kwh = scenario_kwh * (saving_pct / 100.0)
                money_saved_6m = saved_kwh * TARIFF_UAH
                # спрощена оцінка річного ефекту з урахуванням амортизації інвестицій
                net_econ_benefit = (money_saved_6m * 1.8) - (cost / 3.0)
                co2_saved = saved_kwh * CO2_FACTOR + sub['Додатковий_еко_бонус_тонн_CO2'].sum()
                valid_solutions.append({
                    'Код': '+'.join(sub['id']),
                    'Обрані заходи': ', '.join(sub['Назва заходу']),
                    'Бюджет (грн)': cost,
                    'Зниження споживання (%)': saving_pct,
                    'Економія енергії (кВт·год)': round(saved_kwh, 0),
                    'Річний екон. ефект (грн)': round(net_econ_benefit, 0),
                    'Скорочення CO2 (тонн)': round(co2_saved, 2)
                })
    sol_df = pd.DataFrame(valid_solutions)
    if sol_df.empty:
        return None, None
    econ_min, econ_max = sol_df['Річний екон. ефект (грн)'].min(), sol_df['Річний екон. ефект (грн)'].max()
    eco_min, eco_max = sol_df['Скорочення CO2 (тонн)'].min(), sol_df['Скорочення CO2 (тонн)'].max()
    sol_df['Econ_norm'] = (sol_df['Річний екон. ефект (грн)'] - econ_min) / (econ_max - econ_min + 1e-9)
    sol_df['Eco_norm'] = (sol_df['Скорочення CO2 (тонн)'] - eco_min) / (eco_max - eco_min + 1e-9)
    sol_df['Інтегральна_оцінка'] = (1 - eco_weight) * sol_df['Econ_norm'] + eco_weight * sol_df['Eco_norm']
    sol_df = sol_df.sort_values(by='Інтегральна_оцінка', ascending=False).reset_index(drop=True)
    return sol_df.iloc[0], sol_df


print('Каталог заходів:')
measures

In [ ]:
# Таблиця компромісів і аналіз чутливості (КТ 4 / тижні 10–11)
# Крайні рішення: eco_weight=0 (лише економіка) vs eco_weight=1 (лише CO₂)
# + зміна погодного сценарію при фіксованому бюджеті 500 тис. грн

BASE_BUDGET = 500000
compromise_rows = []

scenario_map_cmp = {
    'warm': future_df['forecast_warm_kwh'].sum(),
    'base': future_df['forecast_base_kwh'].sum(),
    'cold': future_df['forecast_cold_kwh'].sum()
}

for scenario_name, scenario_kwh in scenario_map_cmp.items():
    for w, label in [(0.0, 'Лише економіка'), (0.5, 'Компроміс 50/50'), (1.0, 'Лише екологія')]:
        best, _ = optimize_portfolio(BASE_BUDGET, w, scenario_kwh, min_saving_pct=15.0)
        if best is None:
            continue
        compromise_rows.append({
            'Сценарій погоди': scenario_name,
            'Пріоритет': label,
            'eco_weight': w,
            'Код портфеля': best['Код'],
            'Бюджет (грн)': best['Бюджет (грн)'],
            'Зниження (%)': best['Зниження споживання (%)'],
            'Екон. ефект (грн/рік)': best['Річний екон. ефект (грн)'],
            'CO₂ (тонн)': best['Скорочення CO2 (тонн)']
        })

compromise_df = pd.DataFrame(compromise_rows)
print('=== Таблиця компромісів (бюджет 500 000 грн) ===')
print(compromise_df.to_string(index=False))

# Чутливість до бюджету (базовий сценарій погоди, вага 0.5)
sensitivity_rows = []
for budget in [350000, 500000, 650000]:
    best, _ = optimize_portfolio(budget, 0.5, scenario_map_cmp['base'], min_saving_pct=15.0)
    if best is None:
        sensitivity_rows.append({'Бюджет': budget, 'Код': '—', 'Інвестиції': None, 'CO₂ (тонн)': None})
    else:
        sensitivity_rows.append({
            'Бюджет': budget,
            'Код': best['Код'],
            'Інвестиції': best['Бюджет (грн)'],
            'Зниження (%)': best['Зниження споживання (%)'],
            'Екон. ефект (грн/рік)': best['Річний екон. ефект (грн)'],
            'CO₂ (тонн)': best['Скорочення CO2 (тонн)']
        })

sensitivity_df = pd.DataFrame(sensitivity_rows)
print('\n=== Чутливість до бюджету (погода=base, eco_weight=0.5) ===')
print(sensitivity_df.to_string(index=False))
compromise_df

In [ ]:
# ============================================================
# ІНТЕРАКТИВНИЙ БЛОК ДЛЯ ЗАХИСТУ: ЗМІНА ПАРАМЕТРА НАЖИВО
# ============================================================
LIVE_BUDGET = 500000        # наприклад 350000 або 650000
LIVE_ECO_WEIGHT = 0.5       # 0.0 — тільки гроші; 1.0 — тільки CO2
WEATHER_SCENARIO = 'base'   # 'warm' | 'base' | 'cold'

scenario_map = {
    'warm': future_df['forecast_warm_kwh'].sum(),
    'base': future_df['forecast_base_kwh'].sum(),
    'cold': future_df['forecast_cold_kwh'].sum()
}

best_live, all_live = optimize_portfolio(
    budget_limit=LIVE_BUDGET,
    eco_weight=LIVE_ECO_WEIGHT,
    scenario_kwh=scenario_map[WEATHER_SCENARIO],
    min_saving_pct=15.0
)

print('=' * 80)
print(f'ПАРАМЕТРИ ЗАХИСТУ: Бюджет = {LIVE_BUDGET:,} грн | '
      f'Вага екології = {LIVE_ECO_WEIGHT} | Погода = {WEATHER_SCENARIO}')
print('=' * 80)

if best_live is not None:
    print(f"РЕКОМЕНДОВАНЕ РІШЕННЯ: {best_live['Код']} -> {best_live['Обрані заходи']}")
    print(f"Необхідні інвестиції:  {best_live['Бюджет (грн)']:,.0f} грн "
          f"(залишок: {LIVE_BUDGET - best_live['Бюджет (грн)']:,.0f} грн)")
    print(f"Зниження споживання:   {best_live['Зниження споживання (%)']}% "
          f"({best_live['Економія енергії (кВт·год)']:,.0f} кВт·год за 6 міс.)")
    print(f"Річний економ. ефект:  {best_live['Річний екон. ефект (грн)']:,.0f} грн/рік")
    print(f"Скорочення викидів:    {best_live['Скорочення CO2 (тонн)']} тонн CO2")
    print('\nТоп-3 альтернативи:')
    print(all_live[[
        'Код', 'Бюджет (грн)', 'Зниження споживання (%)',
        'Річний екон. ефект (грн)', 'Скорочення CO2 (тонн)', 'Інтегральна_оцінка'
    ]].head(3).round(3).to_string(index=False))
else:
    print('За заданих обмежень допустимих рішень немає. Збільште бюджет або зменште min_saving_pct.')

## Короткі висновки

1. **КТ 1:** дані очищено від пропусків і аномалій; додано `co2_tons` і `cost_uah`.
2. **КТ 2:** багатофакторні моделі краще базової (лише температура); метрики — MAE, RMSE, R².
3. **КТ 3:** сезонний baseline порівняно з тренд+сезонність моделлю; прогноз на 6 міс. з сценаріями warm/base/cold.
4. **КТ 4:** оптимальний портфель залежить від бюджету, ваги екології та погодного сценарію — це зручно демонструвати на захисті.